In [2]:
# ==========================================
# KOF Characters Clustering using NLP + KMeans
# ==========================================

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------
# Load Dataset
# ------------------------------------------

df = pd.read_csv("kof-characters.csv")

# Clean column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

print("\nColumns found:")
print(df.columns.tolist())

# ------------------------------------------
# Required Columns
# ------------------------------------------

required_columns = [
    'name',
    'team',
    'fighting_style',
    'country',
    'bio'
]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"Available columns: {df.columns.tolist()}"
    )

# ------------------------------------------
# Fill Missing Values
# ------------------------------------------

for col in required_columns:
    df[col] = df[col].fillna('').astype(str)

# ------------------------------------------
# Combine Text Features
# ------------------------------------------

df['combined_text'] = (
    df['name'] + ' ' +
    df['team'] + ' ' +
    df['fighting_style'] + ' ' +
    df['country'] + ' ' +
    df['bio']
)

# ------------------------------------------
# TF-IDF Vectorization
# ------------------------------------------

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=1000
)

X = vectorizer.fit_transform(df['combined_text'])

print("\nTF-IDF Matrix Shape:")
print(X.shape)

# ------------------------------------------
# Elbow Method
# ------------------------------------------

inertia = []

for k in range(1, 11):

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    km.fit(X)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia, marker='o')
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(True)
plt.show()

# ------------------------------------------
# Choose Number of Clusters
# ------------------------------------------

k = 5

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

df['cluster'] = kmeans.fit_predict(X)

# ------------------------------------------
# View Cluster Assignments
# ------------------------------------------

print("\nCharacter Clusters:")
print(
    df[['name', 'cluster']]
    .sort_values('cluster')
)

# ------------------------------------------
# Top Terms Per Cluster
# ------------------------------------------

terms = vectorizer.get_feature_names_out()

print("\nTop Terms by Cluster")

for cluster_id in range(k):

    center = kmeans.cluster_centers_[cluster_id]

    top_indices = center.argsort()[-10:][::-1]

    top_terms = [terms[i] for i in top_indices]

    print(f"\nCluster {cluster_id}")
    print(top_terms)

# ------------------------------------------
# PCA Visualization
# ------------------------------------------

pca = PCA(n_components=2, random_state=42)

X_dense = X.toarray()

X_pca = pca.fit_transform(X_dense)

df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df,
    x='PCA1',
    y='PCA2',
    hue='cluster',
    palette='tab10',
    s=100
)

for _, row in df.iterrows():

    plt.text(
        row['PCA1'],
        row['PCA2'],
        row['name'],
        fontsize=8
    )

plt.title("KOF Character Clusters")
plt.legend(title="Cluster")
plt.show()

# ------------------------------------------
# Cluster Summary
# ------------------------------------------

print("\nCluster Sizes:")
print(df['cluster'].value_counts().sort_index())

# ------------------------------------------
# Save Results
# ------------------------------------------

output_file = "kof_characters_clustered.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"\nClustered dataset saved as: {output_file}")


Columns found:
['name', 'gender', 'first_appearance', 'original_game']


ValueError: Missing columns: ['team', 'fighting_style', 'country', 'bio']
Available columns: ['name', 'gender', 'first_appearance', 'original_game']

In [3]:
import pandas as pd

df = pd.read_csv("kof-characters.csv")

print(df.columns.tolist())

['Name', ' Gender', ' First appearance', ' Original game']
